In [1]:
import torch
import sys
from pathlib import Path
import torch, torch.nn as nn
import torchvision.models as models

# Add the project root (parent of NoteBooks) to Python's path
project_root = Path.cwd().parent  # if notebook is in NoteBooks/
sys.path.insert(0, str(project_root))

from src.training.train_cnn import create_cnn_model
from src.models.mlp import create_mlp_model_from_splits
from src.data.loader import *
from src.data.preprocessing import * # need thsi for just a sec. 
from scripts.ensemble_evalute import final_evaluate_mlp_on_test, final_evaluate_cnn_on_test

In [2]:
# constants. 
projections_path = "../IU-xray/indiana_projections.csv"
reports_path = "../IU-xray\indiana_reports.csv"
image_dir = '../IU-xray/images/images_normalized'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = Path("../models")
# where we can find the fetchers later. 
SPLITS_DIR = Path("../data/splits")
MODELS_DIR = Path("../models")

In [3]:
# create the data splits for training and validation purposes. Trust. 
# modes = ["abnormal", "pneumonia"]
# for mode in modes:
#     create_train_test_split(projections_path, reports_path, mode)

In [10]:
# modes = ["abnormal", "pneumonia"]

modes = ["pneumonia"]

for mode in modes:
    df = build_master_df_rich(projections_path, reports_path, mode) # get EVERYTHING. 

    # this should do the trick
    train_df = pd.read_csv(SPLITS_DIR / f"{mode}_train.csv")
    val_df = pd.read_csv(SPLITS_DIR / f"{mode}_val.csv")
    test_df = pd.read_csv(SPLITS_DIR / f"{mode}_test.csv")  # Keep untouched

    # subset = True # this is for prototyping purposes. Just to make sure stuff doesn't crash, not that it works well per se. 
    # if subset == True:
    #     train_df = train_df.sample(n=200, random_state=42)
    #     val_df = val_df.sample(n=200, random_state=42)

    # # 1. Train MLP
    # print("Training MLP's")
    # text_amount = "medium" # use the medium amount of text data
    # model_type = "single" # not ensemble, single mode. 
    # # returns a state dict, not the model
    # MLP_model_state_dict, errors_train, errors_test, losses_train, losses_test = create_mlp_model_from_splits(train_df, val_df, test_df, mode, text_amount, model_type)
    # final_evaluate_mlp_on_test(MLP_model_state_dict, mode, text_amount)

    # 2. Train CNN
    print("training finetuned CNN's sire ")
    num_epochs = 25
    CNN_model_state_dict, history = create_cnn_model(train_df, val_df, image_dir, device=device, mode=mode, num_epochs=num_epochs)
    final_evaluate_cnn_on_test(CNN_model_state_dict, mode, image_dir, device)

    # # save the models...
    # MLP_filename = f"best_mlp_{mode}_{text_amount}.pth"
    # MLP_full_path = save_path / MLP_filename
    # torch.save(MLP_model_state_dict, MLP_full_path)


    CNN_filename = f"best_cnn_densenet_{mode}.pth"
    CNN_full_path = save_path / CNN_filename
    torch.save(CNN_model_state_dict, CNN_full_path)



This is the projections path  ../IU-xray/indiana_projections.csv
Pneumonia mode: 685 positives / 3851 total (17.8%)
training finetuned CNN's sire 
Starting epoch  0
Best new model! AUC 0.5196
Starting epoch  1
Best new model! AUC 0.5624
Starting epoch  2
Best new model! AUC 0.6216
Starting epoch  3
Paitence did not improve
Starting epoch  4
Paitence did not improve
Starting epoch  5
Paitence did not improve
Starting epoch  6
Paitence did not improve
Starting epoch  7
Paitence did not improve
Early stoopping at epoch  7

FINAL CNN TEST EVALUATION - PNEUMONIA

************************************
  t=0.50 | Sensitivity: 36.6% | Specificity: 77.8% | FN: 64 | FP: 105
  t=0.30 | Sensitivity: 87.1% | Specificity: 17.6% | FN: 13 | FP: 389
  t=0.20 | Sensitivity: 97.0% | Specificity: 1.5% | FN: 3 | FP: 465
  t=0.10 | Sensitivity: 100.0% | Specificity: 0.0% | FN: 0 | FP: 472
  t=0.05 | Sensitivity: 100.0% | Specificity: 0.0% | FN: 0 | FP: 472

Test Accuracy : 70.5%
Test AUC-ROC  : 0.6013

Confu

In [ ]:
# code to look at models after they have been run. 

def verify_all_saved_models(modes=["abnormal", "pneumonia"], text_amount="medium"):
    """
    Load every saved model and re-run evaluations to verify they were saved correctly.
    Compares output to ensure consistency.
    """
    import torch
    import pandas as pd
    import numpy as np
    import pickle
    from pathlib import Path
    from sklearn.metrics import roc_auc_score
    
    SPLITS_DIR = Path("../data/splits")
    MODELS_DIR = Path("../models")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    results = {}
    
    print("\n" + "="*80)
    print("MODEL VERIFICATION - Loading all saved models and re-evaluating")
    print("="*80)
    
    for mode in modes:
        print(f"\n{'='*60}")
        print(f"VERIFYING MODE: {mode.upper()}")
        print(f"{'='*60}")
        
        results[mode] = {}
        
        # Load test data
        test_df = pd.read_csv(SPLITS_DIR / f"{mode}_test.csv")
        y_true = test_df['label'].values
        
        # ------------------------------------------------------------
        # 1. Verify MLP
        # ------------------------------------------------------------
        print("\n📦 Loading MLP...")
        mlp_path = MODELS_DIR / f"best_mlp_{mode}_{text_amount}.pth"
        
        if not mlp_path.exists():
            print(f"  ❌ MLP not found at: {mlp_path}")
        else:
            mlp_state_dict = torch.load(mlp_path, map_location=device)
            
            # Check what was saved
            if isinstance(mlp_state_dict, dict):
                if 'network.0.weight' in mlp_state_dict:
                    print(f"  ✅ Valid state dict with {len(mlp_state_dict)} keys")
                else:
                    print(f"  ⚠️ Dict contains unexpected keys: {list(mlp_state_dict.keys())[:5]}")
            else:
                print(f"  ❌ Not a state dict! Type: {type(mlp_state_dict)}")
            
            # Recreate model
            from src.models.mlp import MLP
            input_dim = mlp_state_dict['network.0.weight'].shape[1]
            mlp_model = MLP(input_nodes=input_dim).to(device)
            mlp_model.load_state_dict(mlp_state_dict)
            mlp_model.eval()
            
            # Load vectorizer
            vectorizer_path = MODELS_DIR / f"vectorizer_{mode}_{text_amount}.pkl"
            if vectorizer_path.exists():
                with open(vectorizer_path, 'rb') as f:
                    vectorizer = pickle.load(f)
                print(f"  ✅ Vectorizer loaded: {vectorizer_path}")
            else:
                print(f"  ❌ Vectorizer not found!")
                continue
            
            # Prepare features
            from src.data.nlp_preprocessing import build_nlp_pipeline, prepare_text_features
            nlp = build_nlp_pipeline()
            test_df_clean = prepare_text_features(test_df.copy(), nlp, mode=text_amount)
            X_test = vectorizer.transform(test_df_clean["text_clean"]).toarray()
            
            # Get predictions
            with torch.no_grad():
                X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
                logits = mlp_model(X_tensor).squeeze()
                mlp_probs = torch.sigmoid(logits).cpu().numpy()
            
            mlp_auc = roc_auc_score(y_true, mlp_probs)
            print(f"  📊 MLP Test AUC: {mlp_auc:.4f}")
            
            results[mode]['mlp'] = {
                'auc': mlp_auc,
                'probs': mlp_probs,
                'path': str(mlp_path)
            }
        
        # ------------------------------------------------------------
        # 2. Verify CNN
        # ------------------------------------------------------------
        print("\n🖼️ Loading CNN...")
        cnn_path = MODELS_DIR / f"best_cnn_densenet_{mode}.pth"
        
        if not cnn_path.exists():
            print(f"  ❌ CNN not found at: {cnn_path}")
        else:
            cnn_state_dict = torch.load(cnn_path, map_location=device)
            
            # Check what was saved
            if isinstance(cnn_state_dict, dict):
                if 'features.conv0.weight' in cnn_state_dict:
                    print(f"  ✅ Valid CNN state dict")
                elif 'train_loss' in cnn_state_dict:
                    print(f"  ⚠️ Saved history dict, not state dict!")
                    print(f"     Keys: {list(cnn_state_dict.keys())}")
                else:
                    print(f"  ⚠️ Unknown dict format")
            else:
                print(f"  ❌ Not a dict! Type: {type(cnn_state_dict)}")
            
            # Only proceed if it's a valid state dict
            if isinstance(cnn_state_dict, dict) and 'features.conv0.weight' in cnn_state_dict:
                from src.models.cnn import FinetunedDenseNet
                cnn_model = FinetunedDenseNet().to(device)
                cnn_model.load_state_dict(cnn_state_dict)
                cnn_model.eval()
                
                # Create test loader
                from src.data.dataset import ChestXRayDataset
                from src.data.loader import get_transforms_grayscale
                from torch.utils.data import DataLoader
                
                _, test_transforms = get_transforms_grayscale()
                test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)
                test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
                
                # Get predictions
                cnn_probs = []
                with torch.no_grad():
                    for images, _ in test_loader:
                        images = images.to(device)
                        logits = cnn_model(images).squeeze()
                        probs = torch.sigmoid(logits).cpu().numpy()
                        cnn_probs.extend(probs if hasattr(probs, '__len__') else [probs])
                
                cnn_probs = np.array(cnn_probs)
                cnn_auc = roc_auc_score(y_true, cnn_probs)
                print(f"  📊 CNN Test AUC: {cnn_auc:.4f}")
                
                results[mode]['cnn'] = {
                    'auc': cnn_auc,
                    'probs': cnn_probs,
                    'path': str(cnn_path)
                }
        
        # ------------------------------------------------------------
        # 3. Ensemble (if both models loaded)
        # ------------------------------------------------------------
        if 'mlp' in results[mode] and 'cnn' in results[mode]:
            print("\n🎯 Ensemble Check:")
            ensemble_probs = 0.5 * results[mode]['mlp']['probs'] + 0.5 * results[mode]['cnn']['probs']
            ensemble_auc = roc_auc_score(y_true, ensemble_probs)
            print(f"  Ensemble AUC: {ensemble_auc:.4f}")
            results[mode]['ensemble'] = {'auc': ensemble_auc, 'probs': ensemble_probs}
    
    # ------------------------------------------------------------
    # 4. Summary Table
    # ------------------------------------------------------------
    print("\n" + "="*80)
    print("VERIFICATION SUMMARY")
    print("="*80)
    print(f"{'Mode':<12} {'MLP AUC':<10} {'CNN AUC':<10} {'Ensemble AUC':<12} {'Status':<15}")
    print("-"*80)
    
    for mode in modes:
        mlp_auc = results[mode].get('mlp', {}).get('auc', None)
        cnn_auc = results[mode].get('cnn', {}).get('auc', None)
        ens_auc = results[mode].get('ensemble', {}).get('auc', None)
        
        mlp_str = f"{mlp_auc:.4f}" if mlp_auc else "N/A"
        cnn_str = f"{cnn_auc:.4f}" if cnn_auc else "N/A"
        ens_str = f"{ens_auc:.4f}" if ens_auc else "N/A"
        
        # Status
        if mlp_auc and cnn_auc:
            status = "✅ Complete"
        elif mlp_auc:
            status = "⚠️ CNN missing"
        elif cnn_auc:
            status = "⚠️ MLP missing"
        else:
            status = "❌ No models"
        
        print(f"{mode:<12} {mlp_str:<10} {cnn_str:<10} {ens_str:<12} {status:<15}")
    
    print("\n📁 Model paths checked:")
    print(f"  {MODELS_DIR.absolute()}")
    
    # List all files in models directory
    print("\n📄 Files in models directory:")
    for f in sorted(MODELS_DIR.glob("*")):
        size = f.stat().st_size / 1024
        print(f"  {f.name} ({size:.1f} KB)")
    
    return results


def compare_evaluation_outputs(results_dict):
    """
    Compare the evaluation outputs to ensure they match between training and loading.
    """
    print("\n" + "="*80)
    print("COMPARING EVALUATION OUTPUTS")
    print("="*80)
    
    for mode, data in results_dict.items():
        print(f"\n{mode.upper()}:")
        
        if 'mlp' in data and 'cnn' in data:
            mlp_auc = data['mlp']['auc']
            cnn_auc = data['cnn']['auc']
            ens_auc = data['ensemble']['auc']
            
            # Check if ensemble improves over best single model
            best_single = max(mlp_auc, cnn_auc)
            improvement = ens_auc - best_single
            
            print(f"  MLP:      {mlp_auc:.4f}")
            print(f"  CNN:      {cnn_auc:.4f}")
            print(f"  Ensemble: {ens_auc:.4f}")
            
            if improvement > 0:
                print(f"  ✅ Ensemble improves by +{improvement:.4f}")
            elif improvement > -0.01:
                print(f"  ⚠️ Ensemble similar to best single ({improvement:+.4f})")
            else:
                print(f"  ❌ Ensemble worse than best single ({improvement:+.4f})")
        else:
            print(f"  ⚠️ Missing models for full comparison")


# Run verification
results = verify_all_saved_models(modes=["abnormal", "pneumonia"])
compare_evaluation_outputs(results)


MODEL VERIFICATION - Loading all saved models and re-evaluating

VERIFYING MODE: ABNORMAL

📦 Loading MLP...
  ✅ Valid state dict with 6 keys
  ✅ Vectorizer loaded: ..\models\vectorizer_abnormal_medium.pkl
  📊 MLP Test AUC: 0.8261

🖼️ Loading CNN...
  ✅ Valid CNN state dict
  📊 CNN Test AUC: 0.7440

🎯 Ensemble Check:
  Ensemble AUC: 0.8426

VERIFYING MODE: PNEUMONIA

📦 Loading MLP...
  ✅ Valid state dict with 6 keys
  ✅ Vectorizer loaded: ..\models\vectorizer_pneumonia_medium.pkl
  📊 MLP Test AUC: 0.7951

🖼️ Loading CNN...
  ✅ Valid CNN state dict
  📊 CNN Test AUC: 0.6013

🎯 Ensemble Check:
  Ensemble AUC: 0.7878

VERIFICATION SUMMARY
Mode         MLP AUC    CNN AUC    Ensemble AUC Status         
--------------------------------------------------------------------------------
abnormal     0.8261     0.7440     0.8426       ✅ Complete     
pneumonia    0.7951     0.6013     0.7878       ✅ Complete     

📁 Model paths checked:
  c:\Users\Sean Smith\Documents\GitHub\474-final-project\Not